# Lecture 6: ARIMA and Seasonal ARIMA (SARIMA) Models
In this notebook we will cover:
- Integrated processes and differencing (ARIMA)
- Unit root tests (ADF, KPSS)
- Seasonal ARIMA (SARIMA) models
- Complete modelling workflow on real data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print("Libraries loaded.")


## 6.1 Integrated Processes and Differencing

A process $\{X_t\}$ is an **ARIMA(p,d,q)** process if the $d$-th order differenced series
$$Y_t = (1-B)^d X_t$$
is an **ARMA(p,q)** process.

### The Differencing Operator
- **First difference:** $(1-B)X_t = X_t - X_{t-1}$
- **Second difference:** $(1-B)^2 X_t = X_t - 2X_{t-1} + X_{t-2}$
- **Seasonal difference:** $(1-B^s)X_t = X_t - X_{t-s}$

**When to difference?** When the series is non-stationary (trend or unit root present).


In [ ]:
# ── Visualising the Effect of Differencing ──

np.random.seed(42)
n = 200

# ARIMA(1,1,0) — series requiring first differencing
# True process: Y_t = Y_{t-1} + epsilon_t (random walk)
epsilon = np.random.normal(0, 1, n)
X = np.cumsum(epsilon)  # Random walk = integrated WN

# First and second differences
dX = np.diff(X, n=1)   # (1-B)X_t
ddX = np.diff(X, n=2)  # (1-B)^2 X_t

fig, axes = plt.subplots(3, 1, figsize=(12, 8))
axes[0].plot(X, color='steelblue', lw=1.2)
axes[0].set_title('Original Series $X_t$ — Random Walk (Non-stationary)', fontweight='bold')
axes[0].axhline(X.mean(), color='red', ls='--', alpha=0.5, label=f'Mean={X.mean():.1f}')
axes[0].legend()

axes[1].plot(dX, color='darkorange', lw=1.2)
axes[1].set_title('First Difference $(1-B)X_t$ — Stationary', fontweight='bold')
axes[1].axhline(0, color='red', ls='--', alpha=0.5)

axes[2].plot(ddX, color='green', lw=1.2)
axes[2].set_title('Second Difference $(1-B)^2 X_t$ — Over-differenced', fontweight='bold')
axes[2].axhline(0, color='red', ls='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Original series variance:      {np.var(X):.2f}")
print(f"First difference variance:     {np.var(dX):.2f}")
print(f"Second difference variance:    {np.var(ddX):.2f}")
print("\nNote: Over-differencing increases variance!")


## 6.2 Unit Root Tests

### Augmented Dickey-Fuller (ADF) Test
$$H_0: \phi = 1 \text{ (unit root present, non-stationary)}$$
$$H_1: |\phi| < 1 \text{ (stationary)}$$

**Decision:** If p-value < 0.05, reject $H_0$ => series is stationary.

### KPSS Test
$$H_0: \text{Series is stationary}$$
$$H_1: \text{Unit root present}$$

**Note:** ADF and KPSS should be used together — results may conflict.


In [ ]:
# ── ADF and KPSS Tests ──

def unit_root_tests(series, name="Series"):
    print(f"{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    
    # ADF Test
    adf_result = adfuller(series, autolag='AIC')
    print(f"\nADF Test:")
    print(f"  Statistic  : {adf_result[0]:.4f}")
    print(f"  p-value    : {adf_result[1]:.4f}")
    print(f"  Critical   : 1%={adf_result[4]['1%']:.3f}, 5%={adf_result[4]['5%']:.3f}")
    if adf_result[1] < 0.05:
        print(f"  Decision   : Stationary (H0 rejected)")
    else:
        print(f"  Decision   : NOT Stationary (H0 not rejected)")
    
    # KPSS Test
    kpss_result = kpss(series, regression='c', nlags='auto')
    print(f"\nKPSS Test:")
    print(f"  Statistic  : {kpss_result[0]:.4f}")
    print(f"  p-value    : {kpss_result[1]:.4f}")
    print(f"  Critical   : 5%={kpss_result[3]['5%']:.3f}")
    if kpss_result[1] > 0.05:
        print(f"  Decision   : Stationary (H0 not rejected)")
    else:
        print(f"  Decision   : NOT Stationary (H0 rejected)")

np.random.seed(42)
n = 200
eps = np.random.normal(0, 1, n)

# Three different series
rw = np.cumsum(eps)                         # Random walk
ar1 = np.zeros(n)
for t in range(1, n):
    ar1[t] = 0.8*ar1[t-1] + eps[t]         # Stationary AR(1)

unit_root_tests(rw, "Random Walk (Unit Root Expected)")
unit_root_tests(ar1, "AR(1) phi=0.8 (Stationary Expected)")
unit_root_tests(np.diff(rw), "Differenced RW (Stationary Expected)")


## 6.3 ARIMA(p,d,q) Model Selection

**Practical guide:**
1. Plot ACF/PACF
2. If non-stationary, apply differencing ($d$ = number of differences)
3. From ACF/PACF of the differenced series, determine $p$ and $q$
4. Select model with AICC

| Model | ACF | PACF |
|-------|-----|------|
| AR(p) | Tails off slowly | Cuts off after $p$ lags |
| MA(q) | Cuts off after $q$ lags | Tails off slowly |
| ARMA | Both tail off | Both tail off |


In [ ]:
# ── ARIMA Model Fitting: AirPassengers ──

from statsmodels.datasets import get_rdataset

# Airline passengers dataset
try:
    data = get_rdataset('AirPassengers')
    ap = data.data['value'].values
except:
    # Manual construction (1949-1960 monthly data)
    ap = np.array([112,118,132,129,121,135,148,148,136,119,104,118,
                   115,126,141,135,125,149,170,170,158,133,114,140,
                   145,150,178,163,172,178,199,199,184,162,146,166,
                   171,180,193,181,183,218,230,242,209,191,172,194,
                   196,196,236,235,229,243,264,272,237,211,180,201,
                   204,188,235,227,234,264,302,293,259,229,203,229,
                   242,233,267,269,270,315,364,347,312,274,237,278,
                   284,277,317,313,318,374,413,405,355,306,271,306,
                   315,301,356,348,355,422,465,467,404,347,305,336,
                   340,318,362,348,363,435,491,505,404,359,310,337,
                   360,342,406,396,420,472,548,559,463,407,362,405,
                   417,391,419,461,472,535,622,606,508,461,390,432])

t_idx = pd.date_range('1949-01', periods=len(ap), freq='MS')
ap_series = pd.Series(ap, index=t_idx)

# Log transformation (variance stabilisation)
log_ap = np.log(ap_series)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0,0].plot(ap_series, color='steelblue')
axes[0,0].set_title('Original Series', fontweight='bold')

axes[0,1].plot(log_ap, color='darkorange')
axes[0,1].set_title('Log Transformation', fontweight='bold')

# Seasonal + first difference
diff_log = log_ap.diff(12).diff(1).dropna()
axes[1,0].plot(diff_log, color='green')
axes[1,0].set_title('Log + Seasonal Diff (s=12) + First Diff', fontweight='bold')
axes[1,0].axhline(0, color='red', ls='--', alpha=0.5)

# ACF
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(diff_log, lags=30, ax=axes[1,1], title='ACF — Differenced Log Series')

plt.tight_layout()
plt.show()

# ADF test on differenced series
adf = adfuller(diff_log, autolag='AIC')
print(f"ADF p-value (differenced): {adf[1]:.4f} {'=> Stationary' if adf[1]<0.05 else '=> Not stationary'}")


## 6.4 SARIMA$(p,d,q)(P,D,Q)_s$ Models

The seasonal ARIMA model incorporates both seasonal and non-seasonal components:

$$\Phi_P(B^s)\phi_p(B)(1-B^s)^D(1-B)^d X_t = \Theta_Q(B^s)\theta_q(B)\epsilon_t$$

**Parameter selection:**
- $s$ = seasonal period (12 for monthly, 4 for quarterly)
- $(p,d,q)$ = non-seasonal component
- $(P,D,Q)$ = seasonal component


In [ ]:
# ── SARIMA(1,1,1)(1,1,1)_12 — AirPassengers ──

from statsmodels.tsa.statespace.sarimax import SARIMAX

# Fit SARIMA model
model = SARIMAX(log_ap,
                order=(1,1,1),
                seasonal_order=(1,1,1,12),
                trend='n')
fit = model.fit(disp=False)
print(fit.summary().tables[0])
print(fit.summary().tables[1])

# Residual diagnostics
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
residuals = fit.resid

axes[0,0].plot(residuals, color='steelblue', lw=0.8)
axes[0,0].set_title('Residuals', fontweight='bold')
axes[0,0].axhline(0, color='red', ls='--')

from scipy import stats
stats.probplot(residuals, plot=axes[0,1])
axes[0,1].set_title('QQ Plot', fontweight='bold')

plot_acf(residuals, lags=30, ax=axes[1,0], title='Residual ACF')
plot_pacf(residuals, lags=30, ax=axes[1,1], title='Residual PACF')

plt.tight_layout()
plt.show()

# Information criteria
print(f"\nAIC: {fit.aic:.2f}  |  BIC: {fit.bic:.2f}")


In [ ]:
# ── 24-Month Forecast with SARIMA ──

forecast_steps = 24
pred = fit.get_forecast(steps=forecast_steps)
pred_mean = pred.predicted_mean
pred_ci = pred.conf_int(alpha=0.05)

# Back-transform from log scale
future_idx = pd.date_range(ap_series.index[-1] + pd.DateOffset(months=1),
                            periods=forecast_steps, freq='MS')

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(ap_series, label='Observations', color='steelblue')
ax.plot(future_idx, np.exp(pred_mean), label='Forecast', color='red', lw=2)
ax.fill_between(future_idx,
                np.exp(pred_ci.iloc[:,0]),
                np.exp(pred_ci.iloc[:,1]),
                color='red', alpha=0.2, label='95% CI')
ax.set_title('SARIMA(1,1,1)(1,1,1)_12 — AirPassengers 24-Month Forecast',
             fontweight='bold', fontsize=13)
ax.legend()
ax.set_xlabel('Date')
ax.set_ylabel('Passengers (thousands)')
plt.tight_layout()
plt.show()


## Summary

| Concept | Description |
|---------|-------------|
| **ARIMA(p,d,q)** | ARMA(p,q) applied to the $d$-th differenced series |
| **Unit root** | ADF (H0: unit root), KPSS (H0: stationary) |
| **SARIMA(p,d,q)(P,D,Q)_s** | ARIMA with seasonal component |
| **Seasonal difference** | $(1-B^s)X_t = X_t - X_{t-s}$ |
| **Log transformation** | Variance stabilisation |

**Modelling Steps:**
1. Visualise series — trend/seasonality present?
2. Log transformation (if needed)
3. Unit root test — how many differences?
4. ACF/PACF — select p, q, P, Q
5. Fit with SARIMAX
6. Residual diagnostics — is the model adequate?
7. Produce forecasts
